# Vigía en GPU: experimento con gpt-oss-20b

Antes de correr: **Entorno de ejecución → Cambiar tipo de entorno → GPU L4** (o A100).

El experimento guarda todo en tu Google Drive y se puede reanudar si Colab se desconecta: volvé a correr las celdas desde el principio.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Guardar en Drive (para reanudar)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/vigia_runs/gptoss20b'
import os; os.makedirs(OUT, exist_ok=True); print(OUT)

## 2. Subir `vigia-colab.zip`

Si ya lo subiste a Drive en `MyDrive/vigia-colab.zip`, la celda lo usa sin pedirlo.

In [ ]:
import os, shutil
src = '/content/drive/MyDrive/vigia-colab.zip'
if not os.path.exists(src):
    from google.colab import files
    up = files.upload()
    src = '/content/' + next(iter(up))
shutil.rmtree('/content/vigia', ignore_errors=True)
!unzip -q -o "$src" -d /content/vigia
%cd /content/vigia
!ls

## 3. Dependencias

No reinstala torch: usa el de Colab con CUDA. `kernels` y `triton` habilitan MXFP4 (gpt-oss-20b ocupa ~14 GB).

In [ ]:
!pip install -q "transformers==5.18.0" accelerate safetensors kernels "triton>=3.4" scikit-learn
!pip install -q -e /content/vigia
import torch, transformers; print(torch.__version__, torch.cuda.is_available(), transformers.__version__)

## 4. Prueba corta (5 preguntas)

Confirma que el modelo carga en MXFP4, que los hooks disparan y que la respuesta final sale.

In [ ]:
!python -m vigia.cli evidence --model openai/gpt-oss-20b --device cuda --dtype auto --reasoning-effort low --data data/evidence_v1.jsonl --limit 5 --out /content/smoke --max-tokens 256 --tiers F0,F1 --n-boot 50 2>&1 | tail -20
!python -c "import json; [print(json.loads(l)['response'][:100]) for l in open('/content/smoke/rows.jsonl')]"

## 5. Experimento completo (600 preguntas, ~1-2 h en L4)

Si Colab se corta, volvé a correr las celdas 1 a 3 y esta: retoma donde quedó.

In [ ]:
!python -m vigia.cli evidence --model openai/gpt-oss-20b --device cuda --dtype auto --reasoning-effort low --data data/evidence_v1.jsonl --out "$OUT" --max-tokens 256 --tiers F0,F1,H --n-boot 2000 --bench 4 --bench-runs 5 2>&1 | tail -40

## 6. Resultado

In [ ]:
print(open(OUT + '/summary.txt', encoding='utf-8').read())

In [ ]:
!cd "$OUT" && zip -q -r /content/vigia_gptoss20b_resultados.zip summary.txt report.json report.html detector.json bench.json split.json manifest.json
from google.colab import files; files.download('/content/vigia_gptoss20b_resultados.zip')